# DictaGen lines written by the augmented DiffusionPen word model — PART 3 (the next 10k) — A5000 server

**Part 3:** each style (participants 01-10) writes eligible lines 2,001-3,000 of its own exp39 DictaGen block (shards 8-11 of the style; no overlap with
parts 1-2). Plan and glyph pack: `plan/` of `cyttic/heb-llmgen10k-diffpen-uni10aug` (3 parts, 120 shards; parts 1-2 = the 80 shards already on the Hub, skipped).

Protocol per word (unchanged from parts 1-2): generated by the EMA model of **`cyttic/diffusionpen-uni10aug-words100k`** (DDIM 50 steps, 5 reference words of the style); read by
**exp35** (greedy CTC, letters only); **CER = 0 → accepted**, otherwise generated again; still rejected after **3 generations → the word is removed** from
image and ground truth. Punctuation and digits come from the participant's own font glyphs.

Output: **`cyttic/heb-llmgen10k-diffpen-uni10aug`** (PUBLIC) `data/train-style<k>-shard<8..11>.parquet` + `meta/`. Run: `./run.sh diffpen/gen_llmgen_part3_diffpenaug.ipynb`
(needs `hf auth login` with a write token). Re-run to resume: shards already on the Hub are skipped.

## 1. GPU

In [ ]:
import os, subprocess, sys
os.environ.setdefault('HF_XET_FIXED_UPLOAD_CONCURRENCY', '4')
import torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available()
WORK = os.path.expanduser('~/work'); os.makedirs(WORK, exist_ok=True)

## 2. Dependencies (torch is pinned to the installed build) + upstream DiffusionPen (`unet.py`, `feature_extractor.py`)

In [ ]:
TORCH = torch.__version__.split('+')[0]
open(os.path.join(WORK, 'constraints.txt'), 'w').write(f'torch=={TORCH}\n')
r = subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-c', os.path.join(WORK, 'constraints.txt'), 'torchvision', 'diffusers==0.38.0',
                    'accelerate', 'einops', 'timm', 'safetensors', 'pyarrow', 'hf_xet'], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:]); assert r.returncode == 0
DP = os.path.join(WORK, 'DiffusionPen')
if not os.path.isdir(DP): subprocess.run(['git', 'clone', 'https://github.com/koninik/DiffusionPen.git', DP], check=True)
out = subprocess.run([sys.executable, '-c', 'import torch, torchvision, diffusers; from torchvision.ops import nms; print(torch.__version__, torchvision.__version__, diffusers.__version__, torch.cuda.is_available())'],
                     capture_output=True, text=True); print(out.stdout, out.stderr[-800:]); assert 'True' in out.stdout

## 3. Weights (all from the Hub)

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download, whoami
print('HF user:', whoami()['name'])
STYLE_PATH = hf_hub_download('konnik/DiffusionPen', 'style_models/iam_style_diffusionpen.pth')
STABLE_DIF = snapshot_download('stable-diffusion-v1-5/stable-diffusion-v1-5',
                               allow_patterns=['model_index.json', 'vae/config.json', 'vae/diffusion_pytorch_model.safetensors', 'scheduler/*'])
hf_hub_download('cyttic/diffusionpen-uni10aug-words100k', 'ema_ckpt.pt')
hf_hub_download('cyttic/heb-words100k-uni10aug-diffpen', 'data/train.parquet', repo_type='dataset')
snapshot_download('cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct')
snapshot_download('google/canine-c'); hf_hub_download('timm/mobilenetv2_100.ra_in1k', 'model.safetensors')
print('style encoder:', STYLE_PATH, '| sd15:', STABLE_DIF)

## 4. Worker script (`gen_llmgen10k.py`; only the three repo constants set to the augmented model)

In [ ]:
%%writefile gen_llmgen10k.py
#!/usr/bin/env python3
"""Worker: write the plan's DictaGen lines with the DiffusionPen WORD model, every word verified by exp35 (protocol below), and push each
finished shard of 250 lines as a parquet file to the PUBLIC dataset repo. Self-contained (Kaggle): needs only the upstream DiffusionPen clone
(unet.py, feature_extractor.py) on sys.path and the Hub.

Protocol, per line of the plan (plan/plan.json of the output repo, built by make_llmgen10k_plan.py):
  * every Hebrew-letter run of the line (a word, or the letters of a token such as "בבוקר," -> "בבוקר") is generated by the EMA model of
    cyttic/diffusionpen-uni10-words100k in the line's style (label = style, 5 random reference words of that style from
    cyttic/heb-words100k-uni10-diffpen train, DDIM 50 steps);
  * the image is read by exp35 (cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct, GREEDY CTC, no LM, non-letters of the reading
    ignored). CER == 0 (reading == the word) -> accepted; otherwise it is thrown away and the word is generated again;
  * a word still rejected after --max-tries (3) generations is REMOVED from the line: not drawn and not in the ground truth; the rest of the
    line is kept (punctuation of a removed word moves onto the previous word; a line left with no word is skipped);
  * punctuation and digits come from the participant's own font (plan/glyphs.parquet), word gaps = the participant's space x U(0.8, 1.3);
  * the line is laid out right to left on the shared baseline, cropped to the ink and resized to 64 px high x 22.9 px per character
    (exactly the exp39 human10uni geometry; fit_hhd_geometry.fit can be applied afterwards as for the other sets).
Output rows: image, text (canonical label, without removed words), style (participant_XX), source_text (the plan's line), words (kept),
words_removed, generations.

    python gen_llmgen10k.py --worker 0 --workers 2 --dp /kaggle/working/DiffusionPen --sd <sd15 dir> --style-enc <pth>
"""
import argparse, io, json, math, os, random, re, sys, time

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from PIL import Image, ImageOps

OUT_REPO = "cyttic/heb-llmgen10k-diffpen-uni10aug"
WORD_MODEL = "cyttic/diffusionpen-uni10aug-words100k"
WORD_DS = "cyttic/heb-words100k-uni10aug-diffpen"
READER = "cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct"
CW, CH = 256, 64


# ------------------------------------------------------------------ exp35 reader (= webHebrewOCR/exp32_backend.py, greedy only)
def _sinusoid(n, d):
    pos = torch.arange(n, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
    return pe


class EncoderCTC(nn.Module):
    def __init__(self, vit, n_classes, sub=2, proj=512, seq_layers=0):
        super().__init__()
        self.vit = vit; self.Hd, self.sub, self.proj_dim = vit.config.hidden_size, sub, proj
        self.proj = nn.Sequential(nn.Linear(4 * self.Hd, sub * proj), nn.GELU(), nn.Dropout(0.0))
        self.seq = None
        if seq_layers:
            self.register_buffer("pe", _sinusoid(6 * 24 * sub, proj), persistent=False)
            layer = nn.TransformerEncoderLayer(proj, nhead=8, dim_feedforward=4 * proj, dropout=0.0, activation="gelu", batch_first=True, norm_first=True)
            self.seq = nn.TransformerEncoder(layer, num_layers=seq_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(proj); self.head = nn.Linear(proj, n_classes)

    def forward(self, pixel_values, cols=None):
        t = self.vit(pixel_values=pixel_values).last_hidden_state[:, 1:, :]; B = t.shape[0]
        g = t.reshape(B, 24, 24, self.Hd).reshape(B, 6, 4, 24, self.Hd).permute(0, 1, 3, 2, 4).reshape(B, 6 * 24, 4 * self.Hd)
        y = self.proj(g).reshape(B, 6 * 24 * self.sub, self.proj_dim)
        if self.seq is not None:
            T = y.shape[1]
            pad = (torch.arange(T, device=y.device)[None, :] >= (cols.to(y.device) * self.sub)[:, None]) if cols is not None else None
            y = self.seq(y + self.pe[:T].to(y.dtype), src_key_padding_mask=pad)
        return F.log_softmax(self.head(self.norm(y)).float(), dim=-1)


def reader_preprocess(image):
    image = ImageOps.mirror(image.convert("RGB")); w, h = image.size
    nw = max(1, round(w * 64 / h)); image = image.resize((nw, 64), Image.LANCZOS)
    cont = Image.new("RGB", (384, 384), (255, 255, 255)); arr = np.array(image); sx = dx = dy = 0
    while sx < nw and dy < 384:
        cw = min(nw - sx, 384 - dx); cont.paste(Image.fromarray(arr[:, sx:sx + cw]), (dx, dy)); sx += cw; dx += cw
        if dx >= 384: dx = 0; dy += 64
    t = torch.tensor(np.array(cont), dtype=torch.float32).permute(2, 0, 1) / 255.0
    return (t - 0.5) / 0.5, math.ceil(min(nw, 6 * 384) / 16)


def load_reader(dev):
    from safetensors.torch import load_file
    from transformers import VisionEncoderDecoderConfig, ViTModel
    path = snapshot_download(READER)
    chars = json.load(open(os.path.join(path, "charset.json"), encoding="utf-8"))["chars"]; assert chars[0] == ""
    enc_src = json.load(open(os.path.join(path, "results.json"))).get("encoder_source", "cyttic/trocr-hebrew-matan-exp7")
    cfg = VisionEncoderDecoderConfig.from_pretrained(os.path.dirname(hf_hub_download(enc_src, "config.json"))).encoder
    state = load_file(os.path.join(path, "model.safetensors"))
    seq_layers = len({k.split(".")[2] for k in state if k.startswith("seq.layers.")})
    m = EncoderCTC(ViTModel(cfg, add_pooling_layer=False), len(chars), seq_layers=seq_layers); m.load_state_dict(state, strict=True)
    return m.to(dev).eval(), chars


def ink_crop(im, thr=200):
    a = np.asarray(im.convert("L")) < thr; c = np.where(a.any(0))[0]; r = np.where(a.any(1))[0]
    return im.crop((int(c[0]), int(r[0]), int(c[-1]) + 1, int(r[-1]) + 1)) if len(c) else im


@torch.no_grad()
def read_greedy(model, chars, images, dev, bs=32):
    out = []
    for s in range(0, len(images), bs):
        pre = [reader_preprocess(ink_crop(im)) for im in images[s:s + bs]]
        px = torch.stack([p for p, _ in pre]).to(dev); cols = torch.tensor([c for _, c in pre])
        with torch.autocast("cuda", dtype=torch.float16):
            lp = model(px, cols).float().cpu()
        for j in range(len(pre)):
            ids = lp[j, : int(cols[j]) * 2].argmax(-1).tolist(); prev, txt = 0, []
            for k in ids:
                if k != prev and k != 0: txt.append(chars[k])
                prev = k
            out.append(re.sub(r"[^א-ת]", "", "".join(txt)))
    return out


# ------------------------------------------------------------------ DiffusionPen word model (= test-diff-pen/build_char_bank.load_model)
def load_generator(a, dev):
    from diffusers import AutoencoderKL, DDIMScheduler
    from torchvision import transforms
    from transformers import CanineModel, CanineTokenizer
    sys.path.insert(0, a.dp)
    from feature_extractor import ImageEncoder
    from unet import UNetModel
    vocab = len(json.load(open(hf_hub_download(WORD_DS, "character_classes.json", repo_type="dataset"), encoding="utf-8")))
    args = argparse.Namespace(model_name="diffusionpen", unet="unet_latent", img_size=(64, 256), channels=4, emb_dim=320, num_heads=4,
                              num_res_blocks=1, latent=True, color=True, dataparallel=False, mix_rate=None, interpolation=False, img_feat=True,
                              device=dev, stable_dif_path=a.sd, dataset="iam")
    tok = CanineTokenizer.from_pretrained("google/canine-c")
    te = nn.DataParallel(CanineModel.from_pretrained("google/canine-c"), device_ids=[0]).to(dev)
    unet = UNetModel(image_size=args.img_size, in_channels=4, model_channels=320, out_channels=4, num_res_blocks=1, attention_resolutions=(1, 1),
                     channel_mult=(1, 1), num_heads=4, num_classes=10, context_dim=320, vocab_size=vocab, text_encoder=te, args=args)
    ema = nn.DataParallel(unet, device_ids=[0]).to(dev)
    ema.load_state_dict(torch.load(a.ckpt or hf_hub_download(WORD_MODEL, "ema_ckpt.pt"), map_location=dev, weights_only=False))
    ema.eval().requires_grad_(False)
    vae = AutoencoderKL.from_pretrained(a.sd, subfolder="vae").to(dev).requires_grad_(False)
    ddim = DDIMScheduler.from_pretrained(a.sd, subfolder="scheduler"); ddim.set_timesteps(a.steps)
    fe = ImageEncoder(model_name="mobilenetv2_100", num_classes=0, pretrained=True, trainable=True)
    sd = torch.load(a.style_enc, map_location="cpu", weights_only=False); md = fe.state_dict()
    md.update({k: v for k, v in sd.items() if k in md and md[k].shape == v.shape}); fe.load_state_dict(md)
    fe = fe.to(dev).eval().requires_grad_(False)
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    return ema, vae, ddim, fe, tok, tf


def pad_ref(im):
    im = im.convert("RGB"); w, h = im.size; im = im.resize((max(1, int(w * CH / h)), CH))
    return ImageOps.pad(im, (CW, CH), color=(255, 255, 255)) if im.width <= CW else im.resize((CW, CH))


@torch.no_grad()
def generate(G, words, label, refs, dev, rng):
    ema, vae, ddim, fe, tok, tf = G; n = len(words)
    simgs = torch.stack([tf(r) for _ in range(n) for r in rng.sample(refs, 5)]).to(dev); sfeat = fe(simgs)
    t = tok(words, padding="max_length", truncation=True, return_tensors="pt", max_length=40).to(dev)
    lbl = torch.full((n,), label, dtype=torch.long, device=dev); x = torch.randn((n, 4, CH // 8, CW // 8), device=dev)
    for tt in ddim.timesteps:
        x = ddim.step(ema(x, torch.full((n,), int(tt), dtype=torch.long, device=dev), t, lbl, original_images=simgs, mix_rate=None,
                          style_extractor=sfeat), tt, x).prev_sample
    img = (vae.decode((1 / 0.18215 * x).float()).sample / 2 + 0.5).clamp(0, 1)
    return [Image.fromarray((img[i].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)) for i in range(n)]


# ------------------------------------------------------------------ layout
def canonical(text):
    return re.sub(r"\s+", " ", text.replace("״", '"').replace("׳", "'")).strip()


def to_glyphs(text):
    return canonical(text).replace('"', "״").replace("'", "׳")


def segments(token):
    return re.findall(r"[א-ת]+|[0-9]+|.", token)


def hcrop(im, thr=200):
    g = im.convert("L"); c = np.where((np.asarray(g) < thr).any(0))[0]
    return g.crop((int(c[0]), 0, int(c[-1]) + 1, g.height)) if len(c) else g


MIRROR = {"(": ")", ")": "("}


def compose(text, words, glyphs, who, space, plan, rng):
    """words: accepted 64-px-high word images (baseline 38) in the order of the line's letter runs. Right-to-left layout -> PNG, label."""
    pieces, wi = [], iter(words)                                   # logical order: list of tokens, each a list of strips
    for tok in to_glyphs(text).split(" "):
        strips = []
        for s in segments(tok):
            if re.fullmatch(r"[א-ת]+", s): strips.append(hcrop(next(wi)))
            elif s.isdigit():                                     # digits are left-to-right inside the run
                ds = [rng.choice(glyphs[(who, c)]) for c in s]; w = sum(d.width for d in ds) + 2 * (len(ds) - 1)
                im = Image.new("L", (w, plan["height"]), 255); x = 0
                for d in ds: im.paste(d, (x, 0)); x += d.width + 2
                strips.append(im)
            else: strips.append(rng.choice(glyphs[(who, MIRROR.get(s, s))]))   # right-to-left: a logical "(" is drawn as ")" (bidi mirroring)
        pieces.append(strips)
    vis = []                                                       # visual left-to-right: last token first, runs reversed
    for k, strips in enumerate(reversed(pieces)):
        if k: vis.append(int(round(space * rng.uniform(0.8, 1.3))))
        for j, s in enumerate(reversed(strips)):
            if j: vis.append(2)
            vis.append(s)
    W = sum(v if isinstance(v, int) else v.width for v in vis) + 20
    line = Image.new("L", (W, plan["height"]), 255); x = 10
    for v in vis:
        if isinstance(v, int): x += v
        else: line.paste(v, (x, 0)); x += v.width
    a = np.asarray(line) < 200; ys, xs = np.where(a)
    line = line.crop((max(0, xs.min() - 4), max(0, ys.min() - 4), min(W, xs.max() + 5), min(line.height, ys.max() + 5)))
    label = canonical(text); line = line.resize((int(round(plan["density"] * len(label))), plan["height"]), Image.LANCZOS)
    b = io.BytesIO(); line.save(b, "PNG"); return b.getvalue(), label


# ------------------------------------------------------------------ one shard
def reduce_text(text, keep):
    """Remove the letter runs whose keep flag is False (runs in reading order). A token left with only punctuation is attached to the
    previous token; an empty token disappears. -> glyph-form text (compose() input)."""
    toks, k = [], iter(keep)
    for tok in to_glyphs(text).split(" "):
        segs, had_run, kept_run = [], False, False
        for s in segments(tok):
            if re.fullmatch(r"[א-ת]+", s):
                had_run = True
                if next(k): segs.append(s); kept_run = True
            else: segs.append(s)
        if had_run and not kept_run and not any(x.isdigit() for x in segs):
            if toks and segs: toks[-1] += "".join(segs)
            continue
        if segs: toks.append("".join(segs))
    return " ".join(toks)


def run_shard(sh, G, R, refs, glyphs, plan, a, dev):
    rng = random.Random(f"{plan['seed']}:{sh['style']}:{sh['shard']}"); torch.manual_seed(plan["seed"] * 100 + sh["style"] * 10 + sh["shard"])
    who, label, space = sh["who"], sh["style"], plan["space_px"][sh["who"]]
    lines = []
    for t in sh["texts"]:
        runs = [s for tok in to_glyphs(t).split(" ") for s in segments(tok) if re.fullmatch(r"[א-ת]+", s)]
        lines.append({"text": t, "runs": runs, "img": [None] * len(runs), "tries": [0] * len(runs)})
    n_gen, t0 = 0, time.time()
    for rnd in range(a.max_tries):                                 # round r = try r of every word not accepted yet
        todo = [(li, ri) for li, L in enumerate(lines) for ri in range(len(L["runs"])) if L["img"][ri] is None]
        for s in range(0, len(todo), a.batch):
            part = todo[s:s + a.batch]; words = [lines[li]["runs"][ri] for li, ri in part]
            ims = generate(G, words, label, refs, dev, rng); rd = read_greedy(*R, ims, dev); n_gen += len(part)
            for (li, ri), w, im, r in zip(part, words, ims, rd):
                lines[li]["tries"][ri] += 1
                if r == w: lines[li]["img"][ri] = im.convert("L")
        left = sum(im is None for L in lines for im in L["img"])
        print(f"  style {label} shard {sh['shard']}: try {rnd + 1}/{a.max_tries} done, {len(todo):,} generated, {left:,} words still rejected, "
              f"{(time.time() - t0) / 60:.1f} min", flush=True)
    rows, removed, skipped = [], [], []
    for L in lines:
        keep = [im is not None for im in L["img"]]
        removed += [w for w, k in zip(L["runs"], keep) if not k]
        if not any(keep): skipped.append(L["text"]); continue
        png, lab = compose(reduce_text(L["text"], keep), [im for im in L["img"] if im is not None], glyphs, who, space, plan, rng)
        rows.append({"image": {"bytes": png, "path": None}, "text": lab, "style": who, "source_text": canonical(L["text"]),
                     "words": sum(keep), "words_removed": len(keep) - sum(keep), "generations": sum(L["tries"])})
    return rows, removed, skipped, n_gen, time.time() - t0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--worker", type=int, default=0); ap.add_argument("--workers", type=int, default=1)
    ap.add_argument("--dp", required=True, help="upstream DiffusionPen clone (unet.py, feature_extractor.py)")
    ap.add_argument("--sd", required=True, help="stable-diffusion-v1-5 dir (vae + scheduler)")
    ap.add_argument("--style-enc", required=True, dest="style_enc"); ap.add_argument("--ckpt", default=None, help="local ema_ckpt.pt (default: Hub)")
    ap.add_argument("--plan", default=None, help="local plan dir (default: plan/ of the output repo)")
    ap.add_argument("--word-ds", default=None, dest="word_ds", help="local train.parquet of the word set (default: Hub)")
    ap.add_argument("--steps", type=int, default=50); ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--max-tries", type=int, default=3, dest="max_tries")
    ap.add_argument("--smoke", type=int, default=0, help="N lines of shard --smoke-shard, no push, written to --out")
    ap.add_argument("--smoke-shard", type=int, default=0, dest="smoke_shard")
    ap.add_argument("--out", default="out_llmgen10k"); ap.add_argument("--max-hours", type=float, default=11.0, dest="max_hours")
    a = ap.parse_args(); dev = "cuda:0"; T0 = time.time()
    import pyarrow as pa, pyarrow.parquet as pq
    pdir = a.plan or os.path.dirname(hf_hub_download(OUT_REPO, "plan/plan.json", repo_type="dataset"))
    if not a.plan: hf_hub_download(OUT_REPO, "plan/glyphs.parquet", repo_type="dataset")
    plan = json.load(open(os.path.join(pdir, "plan.json"), encoding="utf-8"))
    glyphs = {}
    for r in pq.read_table(os.path.join(pdir, "glyphs.parquet")).to_pylist():
        glyphs.setdefault((r["who"], r["char"]), []).append(Image.open(io.BytesIO(r["png"])).convert("L"))
    shards = plan["shards"][a.worker::a.workers]
    api = HfApi(); have = set() if a.smoke else set(api.list_repo_files(OUT_REPO, repo_type="dataset"))
    path_of = lambda sh: f"data/train-style{sh['style']}-shard{sh['shard']}.parquet"
    todo = [plan["shards"][a.smoke_shard]] if a.smoke else [sh for sh in shards if path_of(sh) not in have]
    if a.smoke: todo[0] = dict(todo[0], texts=todo[0]["texts"][:a.smoke])
    if not a.smoke: print(f"worker {a.worker}/{a.workers}: {len(shards)} shards, {len(shards) - len(todo)} already on the Hub, {len(todo)} to do", flush=True)
    if not todo: return
    wds = pq.read_table(a.word_ds or hf_hub_download(WORD_DS, "data/train.parquet", repo_type="dataset"), columns=["image", "label"])
    lab = wds.column("label").to_numpy(); refs = {}
    for k in range(10):
        idx = np.random.RandomState(k).choice(np.where(lab == k)[0], 200, replace=False)
        refs[k] = [pad_ref(Image.open(io.BytesIO(wds.column("image")[int(i)].as_py()["bytes"]))) for i in idx]
    del wds
    G = load_generator(a, dev); R = load_reader(dev)
    times = []
    for sh in todo:
        if times and (time.time() - T0 + np.mean(times)) / 3600 > a.max_hours:
            print(f"worker {a.worker}: stopping, the next shard would not fit in {a.max_hours} h (re-run to resume)", flush=True); break
        rows, removed, skipped, n_gen, dt = run_shard(sh, G, R, refs[sh["style"]], glyphs, plan, a, dev); times.append(dt)
        tab = pa.Table.from_pylist(rows); os.makedirs(a.out, exist_ok=True)
        f = os.path.join(a.out, os.path.basename(path_of(sh))); pq.write_table(tab, f)
        n_w = sum(r["words"] for r in rows) + len(removed)
        meta = {"style": sh["style"], "shard": sh["shard"], "lines": len(rows), "lines_skipped": skipped, "words_kept": sum(r["words"] for r in rows),
                "words_removed": removed, "lines_with_removed_word": sum(r["words_removed"] > 0 for r in rows), "generations": n_gen,
                "max_tries": a.max_tries, "minutes": dt / 60}
        json.dump(meta, open(f.replace(".parquet", ".json"), "w"), ensure_ascii=False, indent=1)
        print(f"== style {sh['style']} shard {sh['shard']}: {len(rows)} lines ({len(skipped)} skipped), words removed {len(removed)}/{n_w} "
              f"({100 * len(removed) / max(1, n_w):.1f}%), lines with a removed word {meta['lines_with_removed_word']}, {n_gen:,} generations, "
              f"{dt / 60:.1f} min", flush=True)
        if a.smoke: break
        for attempt in range(5):
            try:
                api.upload_file(path_or_fileobj=f, path_in_repo=path_of(sh), repo_id=OUT_REPO, repo_type="dataset",
                                commit_message=f"style {sh['style']} shard {sh['shard']}")
                api.upload_file(path_or_fileobj=f.replace(".parquet", ".json"), path_in_repo=f"meta/{os.path.basename(f).replace('.parquet', '.json')}",
                                repo_id=OUT_REPO, repo_type="dataset", commit_message=f"meta style {sh['style']} shard {sh['shard']}")
                break
            except Exception as e:
                print("push failed, retrying:", repr(e)[:200], flush=True); time.sleep(30 * (attempt + 1))
    print(f"worker {a.worker}: finished, {(time.time() - T0) / 3600:.2f} h", flush=True)


if __name__ == "__main__":
    main()

## 5. Smoke test — 6 lines of the first part-3 shard (style 0, shard 8), nothing pushed

In [ ]:
r = subprocess.run([sys.executable, 'gen_llmgen10k.py', '--smoke', '6', '--smoke-shard', '80', '--dp', DP, '--sd', STABLE_DIF, '--style-enc', STYLE_PATH,
                    '--batch', '64', '--out', os.path.join(WORK, 'smoke_part3')], capture_output=True, text=True)
print('\n'.join(l for l in (r.stdout + r.stderr).splitlines() if l.startswith(('==', 'Traceback')) or 'Error' in l)); assert r.returncode == 0
import pyarrow.parquet as pq
for row in pq.read_table(os.path.join(WORK, 'smoke_part3', 'train-style0-shard8.parquet')).to_pylist():
    print(row['words_removed'], 'removed |', row['text'])

## 6. Full run — one worker; each finished shard is pushed. Progress is printed every 5 min (log: `~/work/gen_part3.log`)

In [ ]:
import time
log = os.path.join(WORK, 'gen_part3.log')
p = subprocess.Popen([sys.executable, 'gen_llmgen10k.py', '--worker', '0', '--workers', '1', '--dp', DP, '--sd', STABLE_DIF, '--style-enc', STYLE_PATH,
                      '--batch', '64', '--out', os.path.join(WORK, 'out_part3'), '--max-hours', '100'], stdout=open(log, 'w'), stderr=subprocess.STDOUT)
print('worker pid', p.pid, flush=True)
while p.poll() is None:
    time.sleep(300)
    lines = [l.rstrip() for l in open(log) if l.startswith(('==', 'worker', 'Traceback', 'push failed'))]
    print(time.strftime('%H:%M'), lines[-1] if lines else '(loading)', flush=True)
print('exit code', p.returncode); print(''.join(l for l in open(log) if l.startswith(('==', 'worker', 'Traceback'))))

## 7. Check the Hub

In [ ]:
import json
from huggingface_hub import HfApi
files = HfApi().list_repo_files('cyttic/heb-llmgen10k-diffpen-uni10aug', repo_type='dataset')
p3 = sorted(f for f in files if f.startswith('data/') and int(f.split('shard')[1].split('.')[0]) >= 8)
print(len(p3), 'of 40 part-3 shards on the Hub')
tot = {'lines': 0, 'kept': 0, 'removed': 0}
for f in sorted(f for f in files if f.startswith('meta/') and int(f.split('shard')[1].split('.')[0]) >= 8):
    m = json.load(open(hf_hub_download('cyttic/heb-llmgen10k-diffpen-uni10aug', f, repo_type='dataset')))
    tot['lines'] += m['lines']; tot['kept'] += m['words_kept']; tot['removed'] += len(m['words_removed'])
print(tot, f"-> {100 * tot['removed'] / max(1, tot['kept'] + tot['removed']):.1f}% of words removed")
if len(p3) < 40: print('NOT finished — run the notebook again to resume.')